In [1]:
import os
import re
from getpass import getpass

# 1. CONFIGURACIÓN DE LA API DE GROQ
print("=== CONFIGURACIÓN DEL ENTORNO ===")
if 'GROQ_API_KEY' not in os.environ:
    os.environ['GROQ_API_KEY'] = getpass("Introduce tu GROQ API Key (obtenla en console.groq.com): ")
print("✅ API Key configurada correctamente.\n")

# 2. ESCUDO DE SEGURIDAD (PII) - Heredado de tu v4.1
def analizar_seguridad(texto):
    """
    Detecta patrones PII, los anonimiza y devuelve el texto limpio 
    junto con el reporte de hallazgos.
    """
    patrones = {
        "DNI": r"\b\d{8}[A-Z]\b",
        "Teléfono": r"\b[6789]\d{8}\b",
        "Matrícula": r"\b\d{4}[A-Z]{3}\b",
        "IBAN": r"\bES\d{22}\b"
        # Quitamos la póliza de aquí porque la validaremos en su propio estado
    }
    
    hallazgos = []
    texto_limpio = texto
    
    for tipo, regex in patrones.items():
        matches = re.findall(regex, texto_limpio, re.IGNORECASE)
        for m in matches:
            anonimo = m[:2] + "*" * (len(m)-4) + m[-2:]
            hallazgos.append({"tipo": tipo, "anonimo": anonimo})
            texto_limpio = texto_limpio.replace(m, f"[{tipo} PROTEGIDO]")
            
    return hallazgos, texto_limpio

=== CONFIGURACIÓN DEL ENTORNO ===


Introduce tu GROQ API Key (obtenla en console.groq.com):  ········


✅ API Key configurada correctamente.



In [2]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("=== INICIALIZANDO CONOCIMIENTO RAG (FAISS) ===")

# 1. Definición de la Base de Conocimiento con sus Metadatos e IDs Únicos
documentos_segurplus = [
    # ==================== SEGUROS DE AUTOMÓVIL ====================
    Document(
        page_content="Introducción: Los seguros de automóvil de SegurPlus protegen al conductor frente a daños propios, daños a terceros, robo y asistencia en carretera.",
        metadata={"id": "AUT_INTRO_001", "titulo": "Introducción", "compania": "SegurPlus", "tipo_seguro": "auto", "producto": "general", "categoria": "general", "subcategoria": "introduccion", "nivel": "basico", "documento": "catalogo_auto", "version": "1.0"}
    ),
    Document(
        page_content="AUTO BÁSICO (TERCEROS AMPLIADO) - Responsabilidad Civil Obligatoria: Cobertura exigida por la legislación española. Incluye daños materiales y personales causados a terceros.",
        metadata={"id": "AUT_BAS_001", "titulo": "Responsabilidad Civil Obligatoria", "compania": "SegurPlus", "tipo_seguro": "auto", "producto": "auto_basico", "categoria": "coberturas", "subcategoria": "responsabilidad_civil_obligatoria", "nivel": "basico", "documento": "catalogo_auto", "version": "1.0"}
    ),
    Document(
        page_content="AUTO BÁSICO - Responsabilidad Civil Voluntaria y Defensa Jurídica: Amplía los límites legales de responsabilidad civil hasta 50.000.000 €. Incluye Defensa Jurídica con defensa penal y reclamación de daños con un límite de 6.000 €.",
        metadata={"id": "AUT_BAS_002", "titulo": "Responsabilidad Civil Voluntaria", "compania": "SegurPlus", "tipo_seguro": "auto", "producto": "auto_basico", "categoria": "coberturas", "subcategoria": "responsabilidad_civil_voluntaria", "nivel": "basico", "documento": "catalogo_auto", "version": "1.0"}
    ),
    Document(
        page_content="AUTO BÁSICO - Asistencia en Carretera: Disponible desde el kilómetro 0, las 24 horas, los 365 días del año. Incluye: remolque, cambio de rueda y envío de combustible. El tiempo medio estimado de llegada es de 45 minutos en zonas urbanas.",
        metadata={"id": "AUT_BAS_003", "titulo": "Asistencia en Carretera", "compania": "SegurPlus", "tipo_seguro": "auto", "producto": "auto_basico", "categoria": "servicios", "subcategoria": "asistencia_carretera", "nivel": "basico", "documento": "catalogo_auto", "version": "1.0"}
    ),
    Document(
        page_content="AUTO BÁSICO - Robo del Vehículo: Cobertura frente a robo total e intento de robo con daños. Indemnización: Valor a nuevo durante los primeros 24 meses; valor venal mejorado posteriormente.",
        metadata={"id": "AUT_BAS_004", "titulo": "Robo", "compania": "SegurPlus", "tipo_seguro": "auto", "producto": "auto_basico", "categoria": "coberturas", "subcategoria": "robo_vehiculo", "nivel": "basico", "documento": "catalogo_auto", "version": "1.0"}
    ),
    Document(
        page_content="AUTO BÁSICO - Incendio: Cobertura por incendio accidental, explosión y caída de rayo.",
        metadata={"id": "AUT_BAS_005", "titulo": "Incendio", "compania": "SegurPlus", "tipo_seguro": "auto", "producto": "auto_basico", "categoria": "coberturas", "subcategoria": "incendio", "nivel": "basico", "documento": "catalogo_auto", "version": "1.0"}
    ),
    Document(
        page_content="AUTO BÁSICO - Rotura de Lunas: Incluye parabrisas, luneta trasera y ventanillas laterales. Sin franquicia.",
        metadata={"id": "AUT_BAS_006", "titulo": "Rotura de Lunas", "compania": "SegurPlus", "tipo_seguro": "auto", "producto": "auto_basico", "categoria": "coberturas", "subcategoria": "rotura_lunas", "nivel": "basico", "documento": "catalogo_auto", "version": "1.0"}
    ),
    Document(
        page_content="AUTO BÁSICO - Exclusiones Principales: No quedan cubiertos la conducción bajo efectos de alcohol o drogas, participación en competiciones, daños provocados intencionadamente y vehículos sin ITV vigente cuando el siniestro esté relacionado.",
        metadata={"id": "AUT_BAS_007", "titulo": "Exclusiones", "compania": "SegurPlus", "tipo_seguro": "auto", "producto": "auto_basico", "categoria": "exclusiones", "subcategoria": "general", "nivel": "basico", "documento": "catalogo_auto", "version": "1.0"}
    ),
    Document(
        page_content="AUTO PREMIUM (TODO RIESGO) - Daños Propios: Cobertura de daños sufridos por el vehículo incluso cuando el conductor es responsable del accidente. Incluye colisiones, vuelcos y actos vandálicos.",
        metadata={"id": "AUT_PREM_001", "titulo": "Daños Propios", "compania": "SegurPlus", "tipo_seguro": "auto", "producto": "auto_premium", "categoria": "coberturas", "subcategoria": "danos_propios", "nivel": "premium", "documento": "catalogo_auto", "version": "1.0"}
    ),
    Document(
        page_content="AUTO PREMIUM - Vehículo de Sustitución: Disponible cuando la reparación supera las 24 horas o existe robo del vehículo. Duración máxima: 15 días.",
        metadata={"id": "AUT_PREM_002", "titulo": "Vehículo Sustitución", "compania": "SegurPlus", "tipo_seguro": "auto", "producto": "auto_premium", "categoria": "servicios", "subcategoria": "vehiculo_sustitucion", "nivel": "premium", "documento": "catalogo_auto", "version": "1.0"}
    ),
    Document(
        page_content="AUTO PREMIUM - Accidentes del Conductor: Capital asegurado de 50.000 € por fallecimiento y 50.000 € por invalidez permanente.",
        metadata={"id": "AUT_PREM_003", "titulo": "Accidentes del Conductor", "compania": "SegurPlus", "tipo_seguro": "auto", "producto": "auto_premium", "categoria": "coberturas", "subcategoria": "accidentes_conductor", "nivel": "premium", "documento": "catalogo_auto", "version": "1.0"}
    ),
    Document(
        page_content="AUTO PREMIUM - Objetos Personales: Cobertura de equipaje, ordenadores y dispositivos electrónicos dentro del coche con un límite de 2.000 €.",
        metadata={"id": "AUT_PREM_004", "titulo": "Objetos Personales", "compania": "SegurPlus", "tipo_seguro": "auto", "producto": "auto_premium", "categoria": "coberturas", "subcategoria": "objetos_personales", "nivel": "premium", "documento": "catalogo_auto", "version": "1.0"}
    ),
    Document(
        page_content="AUTO PREMIUM - Asistencia Premium: Incluye vehículo de sustitución inmediato, hotel en desplazamientos y transporte alternativo.",
        metadata={"id": "AUT_PREM_005", "titulo": "Asistencia Premium", "compania": "SegurPlus", "tipo_seguro": "auto", "producto": "auto_premium", "categoria": "servicios", "subcategoria": "asistencia_premium", "nivel": "premium", "documento": "catalogo_auto", "version": "1.0"}
    ),
    Document(
        page_content="AUTO PREMIUM - Daños por Fenómenos Naturales y Franquicias: Cobertura por granizo, inundaciones y tempestades cuando no sean asumidos por el Consorcio. Franquicias disponibles a elección del cliente: 150 €, 300 € o 600 €.",
        metadata={"id": "AUT_PREM_006", "titulo": "Fenómenos Naturales", "compania": "SegurPlus", "tipo_seguro": "auto", "producto": "auto_premium", "categoria": "coberturas", "subcategoria": "fenomenos_naturales", "nivel": "premium", "documento": "catalogo_auto", "version": "1.0"}
    ),

    # ==================== SEGUROS DE HOGAR ====================
    Document(
        page_content="Introducción Hogar: SegurPlus ofrece soluciones de protección para viviendas. Los seguros de hogar cubren daños materiales, responsabilidad civil y asistencia urgente en el domicilio.",
        metadata={"id": "HOG_INTRO_001", "titulo": "Introducción Hogar", "compania": "SegurPlus", "tipo_seguro": "hogar", "producto": "hogar_esencial", "categoria": "general", "subcategoria": "introduccion", "nivel": "basico", "documento": "catalogo_hogar", "version": "1.0"}
    ),
    Document(
        page_content="SEGURO HOGAR ESENCIAL - Incendio y explosión: Se cubren los daños ocasionados por incendios accidentales, explosiones de gas doméstico y caída de rayo.",
        metadata={"id": "HOG_ESE_001", "titulo": "Incendio y Explosión", "compania": "SegurPlus", "tipo_seguro": "hogar", "producto": "hogar_esencial", "categoria": "coberturas", "subcategoria": "incendio_explosion", "nivel": "basico", "documento": "catalogo_hogar", "version": "1.0"}
    ),
    Document(
        page_content="SEGURO HOGAR ESENCIAL - Daños por agua: Se cubren daños provocados por rotura accidental de tuberías, fugas en instalaciones fijas y desbordamiento de depósitos domésticos. Límite máximo: 30.000 € por siniestro. No cubre falta de mantenimiento.",
        metadata={"id": "HOG_ESE_002", "titulo": "Daños por Agua", "compania": "SegurPlus", "tipo_seguro": "hogar", "producto": "hogar_esencial", "categoria": "coberturas", "subcategoria": "danos_agua", "nivel": "basico", "documento": "catalogo_hogar", "version": "1.0"}
    ),
    Document(
        page_content="SEGURO HOGAR ESENCIAL - Fenómenos atmosféricos: Se cubren daños causados por lluvia intensa, viento superior a 80 km/h y granizo. Límite máximo de 20.000 €.",
        metadata={"id": "HOG_ESE_003", "titulo": "Fenómenos Atmosféricos", "compania": "SegurPlus", "tipo_seguro": "hogar", "producto": "hogar_esencial", "categoria": "coberturas", "subcategoria": "fenomenos_atmosfericos", "nivel": "basico", "documento": "catalogo_hogar", "version": "1.0"}
    ),
    Document(
        page_content="SEGURO HOGAR ESENCIAL - Robo en la vivienda: Se cubre robo con signos de fuerza y daños en puertas o ventanas. Capital máximo asegurado para contenido: 10.000 €.",
        metadata={"id": "HOG_ESE_004", "titulo": "Robo", "compania": "SegurPlus", "tipo_seguro": "hogar", "producto": "hogar_esencial", "categoria": "coberturas", "subcategoria": "robo_vivienda", "nivel": "basico", "documento": "catalogo_hogar", "version": "1.0"}
    ),
    Document(
        page_content="SEGURO HOGAR ESENCIAL - Responsabilidad Civil Familiar: Cobertura por daños involuntarios causados a terceros (ej. fuga de agua que afecta al vecino, caída de objetos). Límite: 150.000 €.",
        metadata={"id": "HOG_ESE_005", "titulo": "Responsabilidad Civil", "compania": "SegurPlus", "tipo_seguro": "hogar", "producto": "hogar_esencial", "categoria": "coberturas", "subcategoria": "responsabilidad_civil", "nivel": "basico", "documento": "catalogo_hogar", "version": "1.0"}
    ),
    Document(
        page_content="SEGURO HOGAR ESENCIAL - Asistencia Hogar 24 horas: Incluye fontanero, electricista y cerrajero urgente. Tiempo máximo de respuesta: 4 horas en capitales de provincia.",
        metadata={"id": "HOG_ESE_006", "titulo": "Asistencia Hogar", "compania": "SegurPlus", "tipo_seguro": "hogar", "producto": "hogar_esencial", "categoria": "servicios", "subcategoria": "asistencia_24h", "nivel": "basico", "documento": "catalogo_hogar", "version": "1.0"}
    ),
    Document(
        page_content="SEGURO HOGAR ESENCIAL - Exclusiones Principales: Falta de mantenimiento, humedades por condensación, inundaciones del Consorcio, actos intencionados y viviendas deshabitadas más de 90 días.",
        metadata={"id": "HOG_ESE_007", "titulo": "Exclusiones", "compania": "SegurPlus", "tipo_seguro": "hogar", "producto": "hogar_esencial", "categoria": "exclusiones", "subcategoria": "general", "nivel": "basico", "documento": "catalogo_hogar", "version": "1.0"}
    ),
    Document(
        page_content="SEGURO HOGAR PREMIUM - Daños estéticos: Cuando una reparación provoca diferencias visuales, cubre la reposición estética (ej. sustitución completa de azulejos). Límite: 15.000 €.",
        metadata={"id": "HOG_PREM_001", "titulo": "Daños Estéticos", "compania": "SegurPlus", "tipo_seguro": "hogar", "producto": "hogar_premium", "categoria": "coberturas", "subcategoria": "danos_esteticos", "nivel": "premium", "documento": "catalogo_hogar", "version": "1.0"}
    ),
    Document(
        page_content="SEGURO HOGAR PREMIUM - Robo fuera del hogar: Cobertura de bienes robados fuera de la vivienda (ej. robo de bolso, ordenador portátil). Límite: 3.000 € por siniestro.",
        metadata={"id": "HOG_PREM_002", "titulo": "Robo fuera del hogar", "compania": "SegurPlus", "tipo_seguro": "hogar", "producto": "hogar_premium", "categoria": "coberturas", "subcategoria": "robo_fuera_hogar", "nivel": "premium", "documento": "catalogo_hogar", "version": "1.0"}
    ),
    Document(
        page_content="SEGURO HOGAR PREMIUM - Equipos electrónicos: Cobertura de televisores, ordenadores, tablets y consolas. Incluye daños por sobretensión eléctrica con un límite de 8.000 €.",
        metadata={"id": "HOG_PREM_003", "titulo": "Equipos Electrónicos", "compania": "SegurPlus", "tipo_seguro": "hogar", "producto": "hogar_premium", "categoria": "coberturas", "subcategoria": "equipos_electronicos", "nivel": "premium", "documento": "catalogo_hogar", "version": "1.0"}
    ),
    Document(
        page_content="SEGURO HOGAR PREMIUM - Asistencia Informática: Incluye eliminación de virus, configuración de dispositivos y recuperación de datos. Hasta 3 intervenciones anuales.",
        metadata={"id": "HOG_PREM_004", "titulo": "Asistencia Informática", "compania": "SegurPlus", "tipo_seguro": "hogar", "producto": "hogar_premium", "categoria": "servicios", "subcategoria": "asistencia_informatica", "nivel": "premium", "documento": "catalogo_hogar", "version": "1.0"}
    ),
    Document(
        page_content="SEGURO HOGAR PREMIUM - Defensa Jurídica: Reclamación de daños, defensa en conflictos vecinales y asistencia legal telefónica con un límite de 12.000 €.",
        metadata={"id": "HOG_PREM_005", "titulo": "Defensa Jurídica", "compania": "SegurPlus", "tipo_seguro": "hogar", "producto": "hogar_premium", "categoria": "servicios", "subcategoria": "defensa_juridica", "nivel": "premium", "documento": "catalogo_hogar", "version": "1.0"}
    ),
    Document(
        page_content="SEGURO HOGAR PREMIUM - Servicio de Manitas: Incluye montaje de muebles, instalación de cortinas y colocación de estanterías. Hasta 3 servicios al año.",
        metadata={"id": "HOG_PREM_006", "titulo": "Reparaciones", "compania": "SegurPlus", "tipo_seguro": "hogar", "producto": "hogar_premium", "categoria": "servicios", "subcategoria": "reparaciones", "nivel": "premium", "documento": "catalogo_hogar", "version": "1.0"}
    )
]

# 2. Inicializar el modelo de embeddings (gratuito y local)
embeddings_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 3. Construir el almacén de vectores FAISS indexando los documentos
vector_store = FAISS.from_documents(documentos_segurplus, embeddings_model)

print("✅ Base de datos FAISS creada con éxito en memoria.")

C:\Users\jassa\AppData\Local\Temp\ipykernel_13032\3455674431.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


=== INICIALIZANDO CONOCIMIENTO RAG (FAISS) ===


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Base de datos FAISS creada con éxito en memoria.


In [39]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
import unicodedata
from langchain.tools import tool
from pydantic import BaseModel, Field
from typing import Literal

@tool
def mandar_tecnico_hogar(motivo: str) -> str:
    """Simula el envio de un tecnico urgente para una incidencia de hogar."""
    return f"Servicio activado: tecnico de hogar solicitado. Motivo: {motivo}"


@tool
def mandar_grua(motivo: str) -> str:
    """Simula el envio de una grua para una incidencia de coche."""
    return f"Servicio activado: grua solicitada. Motivo: {motivo}"


@tool
def pasar_agente_humano(motivo: str) -> str:
    """Simula la derivacion de la conversacion a un agente humano."""
    return f"Derivacion activada: agente humano solicitado. Motivo: {motivo}"


@tool
def registrar_incidencia(resumen: str) -> str:
    """Simula el registro interno de una incidencia en el sistema."""
    return f"Incidencia registrada correctamente. Resumen: {resumen}"


def decidir_herramienta_operativa(tipo_incidencia, gravedad, texto_usuario):
    texto = texto_usuario.lower()

    if gravedad["nivel"] == "critica":
        return "pasar_agente_humano"

    if any(p in texto for p in ["queja", "reclamacion", "denuncia", "abogado", "muy enfadado"]):
        return "pasar_agente_humano"

    if "hogar" in tipo_incidencia.lower():
        if any(p in texto for p in ["tuberia", "fuga de agua", "inunda", "inundacion", "fontanero", "cerradura", "cerrajero"]):
            return "mandar_tecnico_hogar"

    if "coche" in tipo_incidencia.lower():
        if any(p in texto for p in ["grua", "no arranca", "tirado", "accidente", "rueda", "carretera", "autopista"]):
            return "mandar_grua"

    return "registrar_incidencia"

class AnalisisIncidencia(BaseModel):
    gravedad: Literal["baja", "media", "alta", "critica"] = Field(
        description="Nivel de gravedad detectado en la consulta"
    )
    accion_recomendada: Literal[
        "responder_info",
        "registrar_incidencia",
        "mandar_tecnico",
        "mandar_grua",
        "pasar_agente_humano",
        "emergencias_112"
    ] = Field(description="Accion recomendada para el chatbot")
    requiere_poliza: bool = Field(
        description="Indica si hay que pedir poliza antes de tramitar"
    )
    requiere_humano: bool = Field(
        description="Indica si conviene derivar a un agente humano"
    )
    motivo: str = Field(
        description="Resumen breve del motivo de la decision"
    )

def analizar_incidencia_estructurada(tipo_incidencia, gravedad, texto_usuario):
    texto = texto_usuario.lower()
    tipo = tipo_incidencia.lower()

    requiere_humano = False
    accion = "registrar_incidencia"
    requiere_poliza = True

    if "informacion" in tipo:
        accion = "responder_info"
        requiere_poliza = False

    elif gravedad["nivel"] == "critica":
        accion = "pasar_agente_humano"
        requiere_humano = True

    elif "hogar" in tipo and any(p in texto for p in ["tuberia", "fuga de agua", "inunda", "cerradura", "cerrajero"]):
        accion = "mandar_tecnico"

    elif "coche" in tipo and any(p in texto for p in ["grua", "no arranca", "tirado", "accidente", "rueda"]):
        accion = "mandar_grua"

    if any(p in texto for p in ["queja", "reclamacion", "denuncia", "abogado", "enfadado"]):
        accion = "pasar_agente_humano"
        requiere_humano = True

    return AnalisisIncidencia(
        gravedad=gravedad["nivel"],
        accion_recomendada=accion,
        requiere_poliza=requiere_poliza,
        requiere_humano=requiere_humano,
        motivo=f"Decision basada en gravedad {gravedad['nivel']} y tipo {tipo_incidencia}."
    )

def generar_resumen_incidencia(tipo_incidencia, num_poliza, gravedad, analisis, texto_usuario, resultado_tool=None):
    return {
        "ramo": tipo_incidencia,
        "poliza": num_poliza,
        "gravedad": gravedad["nivel"],
        "accion_recomendada": analisis.accion_recomendada,
        "requiere_humano": analisis.requiere_humano,
        "descripcion_cliente": texto_usuario,
        "accion_interna": resultado_tool if resultado_tool else "Sin accion interna simulada",
        "estado": "pendiente de seguimiento" if analisis.requiere_humano else "gestionado por chatbot"
    }

def formatear_resumen_incidencia(resumen):
    return (
        "[Resumen final de incidencia]"
        f"\n- Ramo: {resumen['ramo']}"
        f"\n- Poliza: {resumen['poliza']}"
        f"\n- Gravedad: {resumen['gravedad']}"
        f"\n- Accion recomendada: {resumen['accion_recomendada']}"
        f"\n- Estado: {resumen['estado']}"
    )

class SegurPlusLLM:
    """Chatbot con LLM (Groq), memoria evolutiva, RAG y deteccion de gravedad."""

    def __init__(self, tipo_incidencia, num_poliza, vector_store):
        self.llm = ChatGroq(
            model_name="llama-3.3-70b-versatile",
            temperature=0.2
        )
        self.history = []
        self.max_history = 5

        self.tipo_incidencia = tipo_incidencia
        self.num_poliza = num_poliza
        self.vector_store = vector_store
        self.resumen_incidencia_actual = None

    def _normalizar_texto(self, texto: str) -> str:
        """Pasa el texto a minusculas y elimina tildes para comparar reglas."""
        texto = texto.lower()
        texto = unicodedata.normalize("NFD", texto)
        texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
        return texto

    def detectar_gravedad(self, user_input: str) -> dict:
        """
        Clasifica la gravedad de la consulta para adaptar tono, longitud y prioridad.
        Inspirado en el patron de clasificacion/ruteo visto en LangGraph, pero
        implementado con reglas simples para mantenerlo transparente en el notebook.
        """
        texto = self._normalizar_texto(user_input)

        marcadores_consulta_actual = [
            "me pasa", "me ha pasado", "he tenido", "tengo", "estoy", "hay",
            "necesito", "urgente", "ahora", "acabo de", "se ha", "hemos"
        ]
        es_modo_info = "informacion" in self._normalizar_texto(self.tipo_incidencia)
        parece_incidencia_actual = any(marcador in texto for marcador in marcadores_consulta_actual)

        reglas = {
            "critica": [
                "no respira", "inconsciente", "sangra mucho", "hemorragia",
                "herido grave", "atrapado", "incendio", "fuego", "explosion",
                "fuga de gas", "electrocutado", "peligro inmediato", "ambulancia",
                "me estoy mareando", "no puedo moverme"
            ],
            "alta": [
                "accidente", "choque", "colision", "autopista", "carretera",
                "grua", "robo", "allanamiento", "cerradura forzada",
                "cristal roto", "inundacion", "fuga de agua", "tuberia rota",
                "no puedo entrar", "no arranca", "me he quedado tirado"
            ],
            "media": [
                "averia", "gotera", "humedad", "rotura", "danos", "daño",
                "reparacion", "tecnico", "parte", "asistencia", "cerrajero",
                "fontanero", "electricista", "lunas"
            ]
        }

        # En consultas informativas, palabras como "ambulancia" o "robo" pueden ser
        # solo dudas sobre coberturas. Solo elevamos gravedad si parece algo actual.
        if es_modo_info and not parece_incidencia_actual:
            nivel = "baja"
            coincidencias = []
        else:
            nivel = "baja"
            coincidencias = []
            for candidato in ["critica", "alta", "media"]:
                coincidencias = [palabra for palabra in reglas[candidato] if palabra in texto]
                if coincidencias:
                    nivel = candidato
                    break

        instrucciones_por_nivel = {
            "critica": {
                "etiqueta": "CRITICA",
                "tono": "calmado, urgente y muy directo",
                "limite": "maximo 3 frases",
                "accion": (
                    "Prioriza la seguridad de la persona. Si hay riesgo vital, heridos, fuego, gas "
                    "o peligro inmediato, indica que llame al 112 ahora. No alargues la respuesta, "
                    "no pidas datos administrativos y confirma la derivacion urgente adecuada "
                    "(ambulancia o agente humano) si procede."
                )
            },
            "alta": {
                "etiqueta": "ALTA",
                "tono": "empatico, resolutivo y breve",
                "limite": "maximo 4 frases",
                "accion": (
                    "Da una accion concreta de asistencia inmediata, como grua, tecnico urgente, "
                    "cerrajero o agente humano. Haz como mucho una pregunta si falta un dato esencial."
                )
            },
            "media": {
                "etiqueta": "MEDIA",
                "tono": "profesional, tranquilizador y claro",
                "limite": "maximo 6 frases",
                "accion": (
                    "Explica el siguiente paso, valida la cobertura con el catalogo y pide un unico "
                    "dato adicional si es necesario para tramitar la incidencia."
                )
            },
            "baja": {
                "etiqueta": "BAJA",
                "tono": "cercano, informativo y ordenado",
                "limite": "respuesta normal y concisa",
                "accion": (
                    "Responde con claridad a la consulta, usando el catalogo oficial y sin activar "
                    "servicios de emergencia salvo que el usuario lo pida explicitamente."
                )
            }
        }

        instrucciones = instrucciones_por_nivel[nivel]
        return {
            "nivel": nivel,
            "etiqueta": instrucciones["etiqueta"],
            "tono": instrucciones["tono"],
            "limite": instrucciones["limite"],
            "accion": instrucciones["accion"],
            "coincidencias": coincidencias
        }

    def obtener_resumen_incidencia(self):
        if not self.resumen_incidencia_actual:
            return "No hay una incidencia registrada para resumir."

        return formatear_resumen_incidencia(self.resumen_incidencia_actual)

    def respond(self, user_input: str) -> str:
        # 1. Determinar el filtro meta para FAISS
        tipo_meta = "auto" if "coche" in self.tipo_incidencia.lower() else "hogar"

        # 2. RAG: Recuperar fragmentos del catalogo
        docs_relevantes = self.vector_store.similarity_search(
            user_input,
            k=3,
            filter={"tipo_seguro": tipo_meta}
        )
        contexto_rag = "\n\n".join([doc.page_content for doc in docs_relevantes])

        # 3. Detectar modo, gravedad y analisis estructurado
        es_modo_info = "informacion" in self._normalizar_texto(self.tipo_incidencia)
        gravedad = self.detectar_gravedad(user_input)

        analisis = analizar_incidencia_estructurada(
            self.tipo_incidencia,
            gravedad,
            user_input
        )

        # 4. Ejecutar herramienta simulada si procede
        resultado_tool = None

        if not es_modo_info and gravedad["nivel"] in ["media", "alta", "critica"]:
            herramienta = decidir_herramienta_operativa(
                self.tipo_incidencia,
                gravedad,
                user_input
            )

            if herramienta == "mandar_tecnico_hogar":
                resultado_tool = mandar_tecnico_hogar.invoke(user_input)
            elif herramienta == "mandar_grua":
                resultado_tool = mandar_grua.invoke(user_input)
            elif herramienta == "pasar_agente_humano":
                resultado_tool = pasar_agente_humano.invoke(user_input)
            else:
                resultado_tool = registrar_incidencia.invoke(user_input)

        # 5. Generar resumen interno de incidencia
        resumen_incidencia = None

        if not es_modo_info:
            resumen_incidencia = generar_resumen_incidencia(
                self.tipo_incidencia,
                self.num_poliza,
                gravedad,
                analisis,
                user_input,
                resultado_tool
            )

        # 6. Definir objetivo segun modo
        if es_modo_info:
            instrucciones_objetivo = """TU OBJETIVO ES MERAMENTE INFORMATIVO:
            - Responde a las dudas del usuario utilizando estrictamente la INFORMACION OFICIAL provista.
            - Explica con claridad que cubre cada producto, sus limites economicos y exclusiones.
            - No ofrezcas enviar servicios de emergencia salvo que el cliente pregunte explicitamente por ese tramite."""
        else:
            instrucciones_objetivo = f"""TU OBJETIVO ES RESOLVER UNA INCIDENCIA ACTIVA:
            - El numero de poliza validado del cliente es: {self.num_poliza}.
            - Determina la asistencia adecuada usando el relato del cliente y el catalogo de coberturas.
            - Opciones: mandar tecnico, mandar grua, llamar ambulancia, pedir taxi o pasar con agente humano.
            - Cuando identifiques el problema, confirma la accion exacta que vas a ejecutar."""

        gravedad_prompt = f"""ADAPTACION POR GRAVEDAD:
        - Gravedad detectada: {gravedad['etiqueta']}.
        - Senales detectadas: {', '.join(gravedad['coincidencias']) if gravedad['coincidencias'] else 'sin senales de urgencia'}.
        - Tono obligatorio: {gravedad['tono']}.
        - Longitud objetivo: {gravedad['limite']}.
        - Prioridad de respuesta: {gravedad['accion']}"""

        analisis_prompt = f"""ANALISIS ESTRUCTURADO INTERNO:
        - Gravedad: {analisis.gravedad}
        - Accion recomendada: {analisis.accion_recomendada}
        - Requiere humano: {analisis.requiere_humano}
        - Motivo: {analisis.motivo}"""

        system_prompt = f"""Eres un agente de asistencia inteligente de SegurPlus.
        Estas en el departamento de: {self.tipo_incidencia.upper()}.

        === INFORMACION OFICIAL DE LAS POLIZAS SEGURPLUS ===
        {contexto_rag}
        ====================================================

        {instrucciones_objetivo}

        {gravedad_prompt}

        {analisis_prompt}

        REGLAS CRITICAS:
        1. Se muy empatico, profesional y directo.
        2. Basate en los datos reales del catalogo.
        3. Haz solo una pregunta a la vez si necesitas mas datos.
        4. NUNCA pidas DNI, telefonos o IBAN.
        5. Si la gravedad es CRITICA, seguridad primero y accion inmediata despues.
        6. No uses Markdown ni formato con asteriscos. Escribe texto plano para consola.
        """

        messages = [SystemMessage(content=system_prompt)]

        for human_msg, ai_msg in self.history[-self.max_history:]:
            messages.append(HumanMessage(content=human_msg))
            messages.append(AIMessage(content=ai_msg))

        messages.append(HumanMessage(content=user_input))

        response = self.llm.invoke(messages)
        ai_response = response.content

      #  if resultado_tool:
       #     ai_response += f"\n\n[Accion interna simulada: {resultado_tool}]"

        if resumen_incidencia:
            prioridad = {"baja": 1, "media": 2, "alta": 3, "critica": 4}

            if self.resumen_incidencia_actual:
                gravedad_anterior = self.resumen_incidencia_actual["gravedad"]
                gravedad_nueva = resumen_incidencia["gravedad"]

                if prioridad[gravedad_anterior] > prioridad[gravedad_nueva]:
                    resumen_incidencia["gravedad"] = gravedad_anterior

                if self.resumen_incidencia_actual["accion_recomendada"] != "registrar_incidencia":
                    resumen_incidencia["accion_recomendada"] = self.resumen_incidencia_actual["accion_recomendada"]

            self.resumen_incidencia_actual = resumen_incidencia
            
        self.history.append((user_input, ai_response))
        return ai_response

In [41]:
import re
import unicodedata


def normalizar_texto_general(texto):
    """Normaliza texto para comparar reglas sin depender de tildes o mayusculas."""
    texto = texto.lower()
    texto = unicodedata.normalize("NFD", texto)
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")

def detectar_triaje_previo(texto):
    texto_norm = normalizar_texto_general(texto)

    patrones_extremos = {
        "bomberos": [
            r"\bhuele.*gas\b",
            r"\bolor.*gas\b",
            r"\bfuga.*gas\b",
            r"\bgas.*cocina\b",
            r"\bgas.*casa\b",
            r"\bgas.*vivienda\b",
            r"\b(casa|cocina|vivienda|piso|garaje)\b.*\b(fuego|llamas|quemando|arde|incendio)\b",
            r"\b(fuego|llamas|quemando|arde|incendio)\b.*\b(casa|cocina|vivienda|piso|garaje)\b",
            r"\bse\s+(me\s+)?esta\s+quemando\b",
            r"\bexplosion\b"
        ],
        "112": [
            r"\bno\s+respira\b",
            r"\binconsciente\b",
            r"\batrapad[oa]\b",
            r"\bherid[oa]\s+grave\b",
            r"\bsangra\s+mucho\b",
            r"\bhemorragia\b",
            r"\belectrocutad[oa]\b",
            r"\bpeligro\s+inmediato\b"
        ]
    }

    patrones_urgentes_no_vitales = [
    r"\btuberia\s+rota\b",
    r"\btuberia.*agua\b",
    r"\bfuga\s+de\s+agua\b",
    r"\bse\s+inunda\b",
    r"\binundando\b",
    r"\bagua\s+por\s+todas\s+partes\b",

    # Cerrajeria / acceso vivienda
    r"\bno\s+puedo\s+entrar.*casa\b",
    r"\bno\s+puedo\s+entrar.*vivienda\b",
    r"\bno\s+puedo\s+entrar.*piso\b",
    r"\bcerradura.*rota\b",
    r"\bcerradura.*bloqueada\b",
    r"\bllaves.*dentro\b",
    r"\bhe\s+perdido\s+las\s+llaves\b",
    r"\bcerrajero\b",

    # Coche / asistencia carretera
    r"\bcoche\s+no\s+arranca\b",
    r"\bvehiculo\s+no\s+arranca\b",
    r"\bme\s+he\s+quedado\s+tirad[oa]\b",
    r"\bnecesito\s+grua\b",
    r"\bpinchazo\b",
    r"\brueda\s+pinchada\b"
]

    for servicio, patrones in patrones_extremos.items():
        coincidencias = [p for p in patrones if re.search(p, texto_norm)]
        if coincidencias:
            return {
                "nivel": "extrema",
                "servicio": servicio,
                "coincidencias": coincidencias
            }

    coincidencias = [p for p in patrones_urgentes_no_vitales if re.search(p, texto_norm)]
    if coincidencias:
        return {
            "nivel": "urgente_no_vital",
            "servicio": "asistencia_segurplus",
            "coincidencias": coincidencias
        }

    return {
        "nivel": "normal",
        "servicio": None,
        "coincidencias": []
    }


def inferir_ramo_desde_texto(texto):
    """Intenta deducir si la incidencia corresponde a Hogar o Coche."""
    texto_norm = normalizar_texto_general(texto)

    claves_hogar = [
        "casa", "hogar", "cocina", "bano", "baño", "salon", "vivienda",
        "tuberia", "agua", "inunda", "cerradura", "puerta", "ventana",
        "gas", "fuego", "incendio"
    ]
    claves_coche = [
        "coche", "vehiculo", "auto", "carretera", "autopista", "grua",
        "rueda", "motor", "no arranca", "accidente", "colision", "choque"
    ]

    if any(clave in texto_norm for clave in claves_hogar):
        return "Hogar"
    if any(clave in texto_norm for clave in claves_coche):
        return "Coche"
    return None


def respuesta_emergencia_extrema(triaje):
    """Respuesta corta cuando la prioridad es seguridad publica, no parte de seguro."""
    if triaje["servicio"] == "bomberos":
        return (
            "Esto es una emergencia. Sal de la zona si puedes hacerlo con seguridad y llama ahora al 112 o a bomberos. "
            "Cuando el peligro haya pasado, vuelve al chat y te ayudo a dar el parte al seguro."
        )
    return (
        "Esto puede ser una emergencia vital. Llama ahora al 112 y sigue sus instrucciones. "
        "Cuando la situacion este controlada, vuelve al chat y tramitamos el parte con tu poliza."
    )


def iniciar_chatbot_segurplus():
    print("\nBienvenido al asistente virtual de SegurPlus")
    print("Estoy aqui para ayudarte con informacion, incidencias y gestiones de tu seguro.")

    # Variables de estado del sistema
    estado_actual = "MENU_PRINCIPAL"
    tipo_incidencia = None
    num_poliza = None
    bot_llm = None

    print("\nChatbot: Hola, bienvenido a SegurPlus. Por favor, selecciona el motivo de tu consulta:")
    print("  1. Quiero informacion de los seguros")
    print("  2. He tenido una incidencia / Necesito asistencia urgente")
    print("  3. Modificacion de Datos / Gestiones Administrativas")

    while True:
        usuario = input("\nTu: ").strip()

        if usuario.lower() in ['salir', 'exit', 'quit']:
            if (
                bot_llm is not None
                and hasattr(bot_llm, "resumen_incidencia_actual")
                and bot_llm.resumen_incidencia_actual
            ):
                print(f"\nChatbot:\n{bot_llm.obtener_resumen_incidencia()}")

            print("\nChatbot: Gracias por confiar en SegurPlus. Que tengas un excelente dia.")
            break

        if not usuario:
            continue

        # PASO A: Anonimizacion de datos (Proteccion PII activa en todo momento)
        alerta_datos, texto_seguro = analizar_seguridad(usuario)
        if alerta_datos:
            print("\n[REPORTE DE PRIVACIDAD]")
            for dato in alerta_datos:
                print(f"  - He ocultado el dato tipo {dato['tipo']} ({dato['anonimo']}) por tu seguridad.")

        # PASO B: Triaje previo. Seguridad primero, tramites despues.
        triaje = detectar_triaje_previo(texto_seguro)
        if triaje["nivel"] == "extrema":
            print(f"\nChatbot: {respuesta_emergencia_extrema(triaje)}")
            continue

        if triaje["nivel"] == "urgente_no_vital" and estado_actual in ["MENU_PRINCIPAL", "CONTEXTO"]:
            ramo_detectado = inferir_ramo_desde_texto(texto_seguro)
            if ramo_detectado:
                tipo_incidencia = ramo_detectado
                estado_actual = "POLIZA"
                print(
                    f"\nChatbot: Entiendo que es urgente, pero no parece una emergencia vital inmediata. "
                    f"Lo tramito como incidencia de {ramo_detectado}. Indica tu numero de poliza de 8 digitos para abrir la asistencia."
                )
                continue
            else:
                estado_actual = "CONTEXTO"
                print("\nChatbot: Entiendo que es urgente. Para tramitarlo, dime de que seguro se trata:")
                print("  1. Seguro de Coche")
                print("  2. Seguro de Hogar")
                continue

        # =========================================================================
        # ESTADO 0: MENU RAIZ (Segmentacion de Intenciones)
        # =========================================================================
        if estado_actual == "MENU_PRINCIPAL":
            if "1" in texto_seguro or "informacion" in normalizar_texto_general(texto_seguro):
                estado_actual = "INFO_TIPO"
                print("\nChatbot: Perfecto. Sobre que ramo de seguros deseas obtener informacion?")
                print("  1. Seguro de Coche")
                print("  2. Seguro de Hogar")

            elif "2" in texto_seguro or "incidencia" in normalizar_texto_general(texto_seguro) or "asistencia" in normalizar_texto_general(texto_seguro):
                estado_actual = "CONTEXTO"
                print("\nChatbot: Lamento escuchar eso. Vamos a gestionar tu asistencia. De que seguro se trata?")
                print("  1. Seguro de Coche")
                print("  2. Seguro de Hogar")

            elif "3" in texto_seguro or "datos" in normalizar_texto_general(texto_seguro) or "gestiones" in normalizar_texto_general(texto_seguro):
                tipo_incidencia = "Gestiones Administrativas"
                estado_actual = "POLIZA"
                print("\nChatbot: Entendido, area de Gestiones Administrativas. Por favor, indicame tu numero de poliza (8 digitos).")
            else:
                print("\nChatbot: Por favor, introduce una opcion valida (1, 2 o 3) para poder derivarte.")

        # =========================================================================
        # ESTADO 1.1: SELECCION DE RAMO PARA CONSULTAS INFORMATIVAS (RAG directo sin poliza)
        # =========================================================================
        elif estado_actual == "INFO_TIPO":
            if "1" in texto_seguro or "coche" in normalizar_texto_general(texto_seguro):
                tipo_incidencia = "Informacion Coche"
                num_poliza = "NO REQUERIDA"
                bot_llm = SegurPlusLLM(tipo_incidencia, num_poliza, vector_store)
                estado_actual = "LLM_CHAT"
                print("\nChatbot: Excelente. Tengo el catalogo de Automoviles cargado. Preguntame lo que quieras sobre coberturas, limites, talleres o exclusiones.")
            elif "2" in texto_seguro or "hogar" in normalizar_texto_general(texto_seguro):
                tipo_incidencia = "Informacion Hogar"
                num_poliza = "NO REQUERIDA"
                bot_llm = SegurPlusLLM(tipo_incidencia, num_poliza, vector_store)
                estado_actual = "LLM_CHAT"
                print("\nChatbot: Excelente. Tengo el catalogo de Hogar listo. Puedes consultarme sobre danos por agua, robos, servicio de manitas, coberturas, etc.")
            else:
                print("\nChatbot: Por favor, selecciona 1 (Coche) o 2 (Hogar) para cargar el catalogo correcto.")

        # =========================================================================
        # ESTADO 1.2: SELECCION DE RAMO PARA INCIDENCIAS (Flujo operativo)
        # =========================================================================
        elif estado_actual == "CONTEXTO":
            if "1" in texto_seguro or "coche" in normalizar_texto_general(texto_seguro):
                tipo_incidencia = "Coche"
                estado_actual = "POLIZA"
                print("\nChatbot: Entendido, apertura de incidencia para Coche. Por favor, indicame tu numero de poliza (8 digitos).")
            elif "2" in texto_seguro or "hogar" in normalizar_texto_general(texto_seguro):
                tipo_incidencia = "Hogar"
                estado_actual = "POLIZA"
                print("\nChatbot: Entendido, apertura de incidencia para Hogar. Por favor, indicame tu numero de poliza (8 digitos).")
            else:
                print("\nChatbot: Por favor, responde con 1 o 2 para identificar el tipo de siniestro.")

        # =========================================================================
        # ESTADO 2: VALIDACION REQUERIDA DE POLIZA (Solo para Incidencias y Gestiones)
        # =========================================================================
        elif estado_actual == "POLIZA":
            match_poliza = re.search(r"\b\d{8}\b", texto_seguro)
            if match_poliza:
                num_poliza = match_poliza.group(0)
                estado_actual = "LLM_CHAT"
                bot_llm = SegurPlusLLM(tipo_incidencia, num_poliza, vector_store)
                print(f"\nChatbot: Poliza {num_poliza} verificada con exito. Cuentame que ha ocurrido para tramitar la asistencia.")
            else:
                print("\nChatbot: Estructura incorrecta. Recuerda que para proceder necesitamos los 8 digitos numericos de tu poliza.")

        # =========================================================================
        # ESTADO 3: CONVERSACION FLUIDA CON LLM + CONTEXTO RAG INYECTADO
        # =========================================================================
        elif estado_actual == "LLM_CHAT":
            respuesta_llm = bot_llm.respond(texto_seguro)
            print(f"\nChatbot: {respuesta_llm}")


# Iniciar la ejecucion de la aplicacion actualizada
iniciar_chatbot_segurplus()



Bienvenido al asistente virtual de SegurPlus
Estoy aqui para ayudarte con informacion, incidencias y gestiones de tu seguro.

Chatbot: Hola, bienvenido a SegurPlus. Por favor, selecciona el motivo de tu consulta:
  1. Quiero informacion de los seguros
  2. He tenido una incidencia / Necesito asistencia urgente
  3. Modificacion de Datos / Gestiones Administrativas



Tu:  exit



Chatbot: Gracias por confiar en SegurPlus. Que tengas un excelente dia.
